# 05 — Hopfield basins and Necker switching

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/05_multistable_dynamics.ipynb)

Companion to §6 of *From Brains to Models*. Training is a flow on a loss landscape. Cognition is trajectories in a recurrent, multistable system. This notebook is a controlled demonstration of that second picture, and on the one formal bridge back to the first: a modern Hopfield update *is* attention.

A 32-unit Hopfield net stores two ±1 patterns. Recall is descent of $E=-\frac12 s^\top W s$; the energy cannot rise. A double-well Langevin walk is the Necker cube as two interpretations and enough noise to switch. One softmax step on the same two patterns is the Ramsauer et al. (2020) link to Figure 1’s attention maps.

No animation, and no open-weights LLM. The sister-repo Hopfield course notebook (capacity, spurious attractors, letter completion) is not repeated here.

**Claim.** Training determines an attractor landscape; inference moves in it; attention is one modern-Hopfield step; bistable perception is two wells and a perturbation.

**Method.** 32-unit Hopfield net, Langevin double-well, one softmax retrieval step. Seed 7.

**Result.** Descent into a stored pattern; energy cannot rise; noise-driven switches; soft vs sharp $\beta$.

**Limit.** Not V1, not binocular rivalry, not a claim that an LLM sees a Necker cube.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Hopfield network](https://en.wikipedia.org/wiki/Hopfield_network).** A recurrent net of simple units that stores patterns as stable states. Given a noisy or partial input it settles into the nearest stored pattern — a *content-addressable memory*, recalled by content rather than by address.
- **Energy, [attractor](https://en.wikipedia.org/wiki/Attractor), basin.** Each state has an "energy"; the network always moves downhill, so stored patterns sit at the bottom of valleys (attractors). The set of starting points that drain into one valley is its *basin*.
- **[Necker cube](https://en.wikipedia.org/wiki/Necker_cube) / bistable perception.** A drawing with two equally valid 3-D readings; perception spontaneously flips between them. Modelled here as two valleys with enough noise to jump between them.
- **Langevin (double-well) dynamics.** Downhill motion plus random jitter — a compact way to capture how a two-state system wanders and occasionally switches.
- **Attention is one Hopfield step.** A modern result: a single step of associative recall is the same operation as the attention in a transformer — the formal bridge from this notebook back to notebook 01.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Energy.** For state $s\in\{\pm 1\}^n$ and symmetric weights $W$, $E(s)=-\tfrac{1}{2}s^\top W s$; asynchronous updates never increase $E$.
- **Storage.** Hebbian outer products $W=\sum_\mu \xi^\mu(\xi^\mu)^\top$ (diagonal zeroed) place patterns $\xi^\mu$ at [attractors](https://en.wikipedia.org/wiki/Attractor).
- **Langevin.** $\dot{x} = -\,U'(x) + \sqrt{2D}\,\eta(t)$ in a double well $U$; the noise $D$ sets the switching rate between wells.
- **Modern Hopfield $=$ [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)).** Retrieval $\xi\leftarrow X\,\mathrm{softmax}(\beta X^\top\xi)$ (Ramsauer et al., 2020); large $\beta$ retrieves one pattern, small $\beta$ mixes — the softmax of attention.
- **Setup.** 32 units, two stored patterns; one softmax step on a $0.55/0.45$ mixture; seed 7.

</details>

The same code writes `paper/figures/multistable_dynamics.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook stores two patterns in a 32-unit Hopfield net, watches recall as energy descent, models the Necker cube as a noisy double well, and shows that one attention step is a Hopfield retrieval. The two blocks below define the recurring terms once, first in plain language and then in notation. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Two pictures.** Training is descent on a *loss* landscape (shaping the network); cognition is a trajectory in the network's *own* state space. This notebook is the second picture, plus the one bridge back to the first.
- **Recall as settling.** Present a corrupted pattern and the network relaxes into the nearest stored memory. Energy cannot rise, so it cannot climb back out on its own.
- **Multistability.** More than one stable state coexists; which one is reached depends on the starting point and on noise. Perception of ambiguous figures behaves this way.
- **Noise-driven switching.** With two wells of similar depth, random fluctuations occasionally push the state over the ridge — the model's analogue of a perceptual flip.
- **Soft versus sharp retrieval ($\beta$).** A temperature knob: low $\beta$ blends stored patterns, high $\beta$ snaps to one. The same knob sets how decisive an attention step is.
- **What this is and is not.** A shared *language* (basins, ridges, switches), not V1, not binocular rivalry, and not a claim that an LLM sees a cube.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Hopfield.** $E(s)=-\tfrac{1}{2}s^\top W s$ with $W$ symmetric and zero diagonal; asynchronous sign updates decrease $E$ monotonically.
- **Capacity.** Two patterns here, well below the $\approx 0.14\,n$ capacity, so recall is clean.
- **Double well.** A potential $U(x)$ with two minima; Langevin dynamics $\dot{x}=-U'(x)+\sqrt{2D}\,\eta$.
- **Retrieval / [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)).** $\xi\leftarrow X\,\mathrm{softmax}(\beta X^\top\xi)$; the $\beta\to\infty$ limit is hard retrieval, matching one-hot attention.
- **Setup.** Seed 7; query $=0.55\,\xi^1 + 0.45\,\xi^2$.
- **Limit.** The retrieving energy $E$, the Langevin potential $U$, and the training loss are three different surfaces; none is a felt state.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.dynamics import (
    attention_retrieve,
    corrupt,
    energy,
    hebb_weights,
    hopfield_async,
    make_patterns,
    necker_energy,
    necker_walk,
    overlaps,
    render_paper_figure,
    run_experiment,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## Two patterns, one energy

Store $\xi_1,\xi_2\in\{\pm 1\}^{32}$ by the Hebb rule $W=\frac1N\sum_\mu \xi^\mu(\xi^\mu)^\top$ with zero diagonal. The energy $E=-\frac12 s^\top W s$ cannot increase under asynchronous sign updates. A cue is $\xi_1$ with 35% of bits flipped. Recall is whoever’s basin the cue lands in.

In [2]:
patterns = make_patterns(n=32, seed=SEED)
W = hebb_weights(patterns)
cue = corrupt(patterns[0], frac=0.35, seed=SEED + 3)
recalled, path, energies = hopfield_async(cue, W, n_sweeps=6, seed=SEED)
print("stored energies ", energy(patterns[0], W), energy(patterns[1], W))
print("cue     overlaps", overlaps(cue, patterns), "E", energy(cue, W))
print("recalled overlaps", overlaps(recalled, patterns), "E", energies[-1])
print("energy never rose?", all(energies[i] >= energies[i + 1] - 1e-12 for i in range(len(energies) - 1)))

stored energies  -15.0 -15.0
cue     overlaps [ 0.3125 -0.0625] E -0.625
recalled overlaps [1. 0.] E -15.0
energy never rose? True


## Necker as a two-well

$E(x)=\frac14(x^2-1)^2$ has wells at $x=\pm 1$ and a saddle at $0$. Langevin dynamics $dx=-E'(x)\,dt+\sigma\,dW$ dwells in a well and, with enough $\sigma$, crosses. The two wells are the two interpretations of a Necker cube. The walk is not a person seeing a cube. It is the *shape* of bistable perception: two basins, a perturbation.

In [3]:
walk = necker_walk(seed=SEED)
sign = (walk > 0).astype(int)
print(f"steps {len(walk)}  sign flips {int((sign[1:] != sign[:-1]).sum())}  mean {walk.mean():+.2f}")
print(f"E at wells {float(necker_energy(-1.0)):.3f} {float(necker_energy(1.0)):.3f}  E at saddle {float(necker_energy(0.0)):.3f}")

steps 1600  sign flips 18  mean -0.37
E at wells 0.000 0.000  E at saddle 0.250


## Attention as one Hopfield step

Ramsauer et al.: $\xi \leftarrow X\,\mathrm{softmax}(\beta X^\top \xi)$. That is attention with keys $=$ values $=$ the stored patterns, and one query. Soft $\beta$ splits the mass; sharp $\beta$ retrieves a single key. The query below is a bit closer to $\xi_1$ than to $\xi_2$. This is not notebook 01’s transformer. It is the *update* that notebook 01’s attention maps are a picture of.

In [4]:
query = 0.55 * patterns[0] + 0.45 * patterns[1]
query = query / np.linalg.norm(query)
for beta in (1.0, 8.0):
    retrieved, w = attention_retrieve(query, patterns, beta)
    print(f"β={beta:.0f}  weights {w[0]:.3f} {w[1]:.3f}  overlap-with-ξ1 {overlaps(np.sign(retrieved), patterns)[0]:+.2f}")

β=1  weights 0.689 0.311  overlap-with-ξ1 +1.00
β=8  weights 0.998 0.002  overlap-with-ξ1 +1.00


## Paper figure

A is the landscape and the recall path. B is the theorem (energy falls). C is switching, not settling. D is the one-step bridge to attention.

In [5]:
result = run_experiment(seed=SEED)
fig = render_paper_figure(result)
plt.show()
write_figure(result, root / "paper" / "figures")

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49889/3996014821.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/multistable_dynamics.pdf
wrote <checkout>/paper/figures/multistable_dynamics.png


PosixPath('<checkout>/paper/figures/multistable_dynamics.pdf')

## What to notice

**Descent is not the same achievement as remembering the right thing.** Panel B proves the energy fell. Panel A shows *which* well it fell into. A spurious attractor would also make B look like success. The sister-repo Hopfield notebook is built on that gap. Here the two patterns are few enough, and orthogonal enough, that the cue reaches $\xi_1$. That is a writing decision, not a capacity theorem.

**Training and inference are two landscapes.** SGD on a transformer writes the keys. The energy in A is not that loss. Attention in D *reads* what was written, in one softmax. Ramsauer et al. are the formal link; notebook 01 is the picture of the softmax.

**Panel C is a perturbation, not a cube.** The wells are $\pm 1$; the colour is time. An ambiguous prompt is a cousin only at this level of grain: two completions, a shared prefix, noise or a rewording as the perturbation. The paper does not say a language model *is* a Necker cube.

**Itinerancy is missing on purpose.** Winnerless competition (Rabinovich, Kelso) is the regime where the state does not settle. Sequential thought lives there. A static figure can show a switch. It cannot show a tour. The animated version is notebook 05b.

## Why this example is here

This notebook is not a tutorial on associative memory, and it is not a model of a Necker cube in V1. It is a controlled demonstration of the one fact §6 of the paper needs: **training determines an attractor landscape; inference moves in it; attention is one modern-Hopfield step; bistable perception is two wells and a perturbation.**

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never met a Hopfield net. The two readings below are of the *same* object. Nothing in the code is a patient, and nothing is a theorem about consciousness. The next section will need basins and ignition; this one only makes “basin” something you can point at.

### What each panel is asked to do

A 32-dimensional state $s\in\{\pm 1\}^{32}$ has energy $E=-\frac12 s^\top W s$ with $W$ the Hebb matrix of two patterns. Asynchronous updates take $s_i\leftarrow \mathrm{sign}((Ws)_i)$. Hopfield’s theorem: $E$ is nonincreasing. The plane in A is the span of $\xi_1,\xi_2$; a point $(u,v)$ is $\tanh$ of the corresponding linear combination. The rust path is a 35\%-corrupted $\xi_1$ written back into that plane after every few flips.

The Necker walk is overdamped Langevin on $E(x)=\frac14(x^2-1)^2$:
$$
dx=(x-x^3)\,dt+\sigma\,dW_t.
$$
Wells at $\pm 1$, saddle at $0$. The attention step is
$$
\xi\leftarrow X\,\mathrm{softmax}(\beta X^\top \xi),
$$
with $X=(\xi_1,\xi_2)$ and one query slightly closer to $\xi_1$.

**For the clinician and the neuroscientist.** The motivation is a kind of memory you already trust, and a kind of seeing you have already watched flip. You recognise a colleague from the back of the head, from three notes of a laugh, from a handwritten *th*. You did not supply an address, and nothing looked anything up. That is *content-addressable* recall: the fragment is enough, and what comes back is not the fragment but the person. Hopfield’s claim is that the network does not search a filing cabinet. It *falls*. The stored people are valleys; the laugh is a starting point on the slope; recall is gravity. A differential diagnosis that “clicks” after two extra signs is the same shape at the bedside — not because cortex is a 32-unit Hopfield net, but because the interesting event is *completion toward a familiar whole*, not lookup of a code.

A Necker cube is the same claim about perception. The ink does not change. What changes is which well you are in: cube-from-above or cube-from-below. Rivalry, ambiguous figures, and some clinical oscillations (a tremor that jumps between two frequencies, a percept that will not stay put) are cousins at this grain: two interpretations, a shared stimulus, a perturbation. The walk in panel C is not binocular rivalry and it is not a patient. It is the schematic of *why a stable drawing can yield an unstable experience* — the instability is in the state, not in the page.

An LLM that completes the same prefix two different ways is being scored, in this schematic, as a walk that left one well for the other. A slight rewording of a prompt is a perturbation; a temperature sample is a perturbation; an ambiguous clinical letter (“the patient was started on …”) is a prefix that sits near a saddle. That is not a claim that the model *sees a cube*, and it is not a claim that next-token prediction is perception. It is a claim that the *shape* — two basins, a shared start, a perturbation — is what §6 needs in order to talk about mode-switching without appealing to mystery. Fluent self-report in the next section is unsurprising for the same reason a well is easy to fall into: the corpus already dug it.

What this perspective is *for*: to stop treating “the model retrieved a memory” and “the model changed its mind” as two unrelated mysteries. Both are motion on a landscape. What it is *not* for: locating a lesion, timing an ERP, or saying which well a particular patient is in. Those need data this notebook does not have.

**For the mathematically inclined.** The motivation is to put three standard objects on one page so the paper’s metaphor is a theorem, an SDE, and a softmax, not a loose analogy. They are not the same dynamical system. They share a *vocabulary*: a function that goes downhill, more than one local minimum, and a parameter that either forbids or permits crossing.

The discrete net is a Lyapunov argument on the hypercube $\{\pm 1\}^N$. With $W=W^\top$ and $W_{ii}=0$,
$$
E(s)=-\tfrac12 s^\top W s
$$
is a Lyapunov function for the asynchronous map $s_i\leftarrow\mathrm{sign}((Ws)_i)$: each accepted flip strictly decreases $E$ (or leaves it unchanged if the local field was zero). Every trajectory therefore ends at a fixed point. Two-cycles are possible for some $W$; the zero-diagonal Hebb rule on these two patterns does not produce one, which you can read off panel B’s plateau. Descent is not the same achievement as *correct* recall. A spurious attractor is also a fixed point, and $E$ would fall into it just as obediently. Capacity $\approx 0.138N$ and the mixture states of the sister-repo notebook are that caveat made quantitative. Here $P=2\ll N$, so the interesting event is the theorem, not the overload.

The plane in A is not $E$ on $\mathbb{R}^2$. It is $E\circ\iota$, where $\iota(u,v)=\tanh\bigl((u\xi_1+v\xi_2)/\|u\xi_1+v\xi_2\|\cdot 3\bigr)$ embeds the two-pattern span into $[-1,1]^N$. Contours are therefore a *view* of $E$, not a change of the state space. The rust path is the async trajectory projected by least squares back onto that span. If the path looks jagged, that is the discrete flips, not a stiff ODE.

The Necker walk is overdamped Langevin on a double well,
$$
dx=-E'(x)\,dt+\sigma\,dW_t,\qquad E(x)=\tfrac14(x^2-1)^2,
$$
so $-E'(x)=x-x^3$. Wells at $\pm 1$, barrier $\Delta E=1/4$ at the saddle $x=0$. Kramers’ mean time to switch scales as $e^{2\Delta E/\sigma^2}$ in the small-noise limit; we do not compute it. Eighteen sign flips at this $\sigma$ are a sample path, not a rate. The SDE is on $\mathbb{R}$, not on the hypercube: panel C is an *analogue* of bistability, not a continuum limit of panel A.

The attention step is exact and untrained:
$$
w=\mathrm{softmax}(\beta X^\top q)\in\Delta^{1},\qquad \hat\xi=w^\top X.
$$
There is no $Q,K,V$ factorisation, no $\sqrt{d}$, no causal mask, no SGD. $\beta$ is an inverse temperature. Soft $\beta$ yields a mixture; large $\beta$ concentrates on $\arg\max_i \xi_i^\top q$. Ramsauer et al. are the statement that a transformer attention head *is* this update with $X$ learned. Notebook 01 is a picture of $w$ after that learning. This cell is the update with $X$ handed to you.

What this perspective is *for*: to keep “energy,” “well,” and “attention” from floating free of a state space. What it is *not* for: a convergence rate for SGD, a spectral analysis of $W$, or a proof that language-model modes are Kramers crossings.

### The four panels

**A — the fragment found the person.** Two valleys in the plane of $\xi_1,\xi_2$; a corrupted $\xi_1$ walks into $\xi_1$. The white dots are the memories we stored, not ones the net invented.

**For the clinician and the neuroscientist.** This is the laugh becoming the person. The starting rust dot is a partial history, a poor-quality image, a few letters of a name. The teal end is the familiar whole. If the path had ended in the other valley, the fragment would have *mis*-recognised — a confabulation with the same mechanism as a hit. Vividness is a property of the valley, not evidence about the past.

**For the mathematically inclined.** Contours of $E\circ\iota$ plus a projected async trajectory. The axes are least-squares coefficients in $\mathrm{span}\{\xi_1,\xi_2\}$, not the overlaps $m=\frac1N Xs$ (those are close when the patterns are nearly orthogonal, and they are not identical). Dark is low $E$.

**B — it got more settled, which is not the same sentence.** Energy versus update. The drop is the theorem; the plateau is arrival.

**For the clinician and the neuroscientist.** A falling score is not insight. It is “the story stopped moving.” A confident wrong diagnosis also stops moving. Read B only after A has told you *which* story.

**For the mathematically inclined.** The restriction of $E$ to the async path. Monotonicity is Hopfield’s Lyapunov argument; the numerical check `energy never rose? True` is that argument on this $W$ and this cue. The plateau at $-15$ is $E(\xi_1)=E(\xi_2)$ for these two Hebb-stored patterns, not a universal constant.

**C — the cube flipped.** A W-shaped $E(x)$; points coloured by time sit in one well, then the other.

**For the clinician and the neuroscientist.** The page did not change. The state did. That is the whole of bistable perception at this grain. A prompt that can be completed two ways is the linguistic cousin: the prefix is the page; the sample is the kick. Do not take the wells as two neurons, two hemispheres, or two diagnostic codes.

**For the mathematically inclined.** One sample path of the SDE above, plotted as $(x_t,E(x_t))$ and coloured by $t$. The dotted lines are the deterministic minima. Crossing count is a path functional, not Kramers’ time.

**D — looking harder at the closer match.** Softmax weights on the two keys, at two $\beta$.

**For the clinician and the neuroscientist.** A soft glance still sees both faces in the crowd. A sharp glance commits. Temperature in a language model is the same knob: it does not add knowledge; it changes how winner-take-all the retrieval is. Instruction-tuning, later in the paper, is among other things a way of digging one well deeper than the other.

**For the mathematically inclined.** Two points of $\Delta^1$. If $s_i=\xi_i^\top q$, then $w_i=e^{\beta s_i}/\sum_j e^{\beta s_j}$. As $\beta\to\infty$, $w\to e_{i^\star}$ for $i^\star=\arg\max s_i$ (ties broken by the implementation). As $\beta\to 0$, $w\to (1/2,1/2)$. The retrieved vector $\hat\xi=w^\top X$ is a convex combination of the keys; $\mathrm{sign}(\hat\xi)$ happened to equal $\xi_1$ at both $\beta$ we plotted, because the query was already on that side of the bisector.

### What this is not

It is not the $0.138N$ capacity curve, not a letter-completion demo, not STDP, not a recurrent cortical column, and not a claim that GPT-4 has two wells. Itinerant sequential dynamics are named in the paper and omitted from the figure. Section 7 will need a bottleneck and an ignition, not a Hopfield net.

### What this has to do with being human

Section 2’s transformer already *looks* like retrieval: attention is a weighted glance at the past. This notebook is why that glance has a physics. A person who recognises a face from a fragment, who cannot hold both Necker readings at once, and who can be tipped from one plan to another by a slight rewording, is being described — at this grain — as a system with more than one well. A foundation model can be described the same way without thereby acquiring a body, a cost for being wrong, or an experience of the flip. The remainder the paper is after sits in that gap: the wells we can share as mathematics, and the viability constraints we cannot.

Change `SEED` or $\sigma$ and the rust path and the yellow crossings will move. The claim should not: **two stored patterns, descent that cannot rise, two wells that can switch, one softmax that is attention.**

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. Content-addressable recall: a fragment falls into a whole ([Hopfield](https://en.wikipedia.org/wiki/Hopfield_network)). When is a “click” in diagnosis the same computational shape, and when is vividness a spurious [attractor](https://en.wikipedia.org/wiki/Attractor)?

2. Energy never rises. Why is a falling score not insight — in this net, and in a confident wrong diagnosis?

3. Two wells, one page ([Necker cube](https://en.wikipedia.org/wiki/Necker_cube)). What does bistable perception require that a changing stimulus does not? What does the SDE add that the discrete net does not?

4. [Attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) as one modern-Hopfield step: $\beta$ is inverse temperature. What clinical “commitment” is a large $\beta$, and what knowledge does it not add?

5. Training determines the landscape; inference moves in it. Map that split onto learning versus seeing in a person, and onto a weight update versus a forward pass in a net.

6. A prompt completed two ways is a walk that left one well. Why is that not a claim that the model sees a cube?

7. Energy, well, and perturbation are a shared dynamical vocabulary. Which of Marr’s rows is that vocabulary, and which row is still open?

8. The wells we can share as mathematics do not include a viability constraint. State one constraint that a Kramers crossing does not contain. ([Allostasis](https://en.wikipedia.org/wiki/Allostasis).)

9. Corrupt a pattern until recall fails. What does capacity tell you about a memory that “feels complete”?

10. Noise can switch the well. In a note that can be read two ways, what is the noise, and what is the page?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Hopfield, J. J. (1982).** [*Neural networks and physical systems with emergent collective computational abilities*](https://redwood.berkeley.edu/wp-content/uploads/2018/08/hopfield82.pdf). *PNAS* 79(8), 2554–2558. Opened PDF.
   - *(i) From brains to models.* Content-addressable memory as a [phase-space flow](https://en.wikipedia.org/wiki/Attractor): a large net of simple units settles to a stored pattern from any sufficient fragment. Energy cannot rise. Panels A–B are that claim at 32 units. The opened abstract also lists generalisation, familiarity, and error correction as emergent collective properties.
   - *(ii) What AI reveals about being human.* A [Hopfield](https://en.wikipedia.org/wiki/Hopfield_network) well is a memory, not a percept of a cube and not a mood. The teaching move is to share the *language* (basins, perturbations) without promoting the toy to cortex.

2. **Ramsauer, H., et al. (2020).** [*Hopfield Networks is All You Need*](https://arxiv.org/abs/2008.02217). arXiv:2008.02217.
   - *(i) From brains to models.* A modern Hopfield update *is* [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)): $\xi \leftarrow X\,\mathrm{softmax}(\beta X^\top\xi)$. Panel D is one such step. Soft $\beta$ splits the mass; sharp $\beta$ retrieves one key. Training determined the keys; this step only retrieves.
   - *(ii) What AI reveals about being human.* The formal bridge does not make a transformer a Hopfield brain. The loss that shaped the keys is not the energy that retrieves them, and neither is a felt interpretation of a [Necker cube](https://en.wikipedia.org/wiki/Necker_cube).

3. **Vaswani, A., et al. (2017).** [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). *NeurIPS* 30. arXiv:1706.03762.
   - *(i) From brains to models.* The architecture whose attention step Ramsauer et al. identify with modern Hopfield retrieval. Notebook 01’s maps are that step on characters; panel D is the same equation on two stored patterns.
   - *(ii) What AI reveals about being human.* Retrieval in a landscape of keys is not bistable perception. An ambiguous prompt is analogous *at the level of basins*. It is not a cube in visual cortex.

4. **Necker, L. A. (1832).** [*Observations on some remarkable optical phaenomena…*](https://en.wikipedia.org/wiki/Necker_cube). Entry via the opened Necker-cube page.
   - *(i) From brains to models.* The [wire-frame cube](https://en.wikipedia.org/wiki/Necker_cube) that supports two consistent 3-D interpretations and spontaneous reversals. Panel C is that fact as a double well plus Langevin noise — a teaching energy, not a measured cortical potential.
   - *(ii) What AI reveals about being human.* People see one interpretation at a time and feel the switch. The toy switches a scalar $x$. Theme (ii) is the experience of reversal, which no well in $E(x)$ reports.

5. **Elhage, N., et al. (2021).** [*A Mathematical Framework for Transformer Circuits*](https://transformer-circuits.pub/2021/framework/index.html). *Transformer Circuits Thread*.
   - *(i) From brains to models.* Attention as a residual write: each head retrieves and adds. Combined with Ramsauer et al., the [mechanistic](https://en.wikipedia.org/wiki/Mechanistic_interpretability) story and the dynamical story meet at one softmax. They still describe different objects (a circuit vs a flow).
   - *(ii) What AI reveals about being human.* A circuit that retrieves is not a trajectory that lives. Do not collapse interpretability into dynamics or dynamics into experience.

6. **Doerig, A., et al. (2023).** [*The neuroconnectionist research programme*](https://konklab.fas.harvard.edu/Papers/Doerig_2023_NRN.pdf). *Nature Reviews Neuroscience* 24, 431–450.
   - *(i) From brains to models.* Multistable [attractors](https://en.wikipedia.org/wiki/Attractor) are one of the computational languages ANNs can state. This notebook states a small one so it can be run. The programme’s test is whether the statement generates a comparison that can fail.
   - *(ii) What AI reveals about being human.* Winnerless competition and itinerancy (the paper’s broader dynamical vocabulary) are where sequential thought may live. Figure 4 shows settling and switching, not itinerancy. Name that limit.


### Exercises

1. **Corrupt until it fails.** `corrupt(pattern, frac)` flips a fraction of bits, then `hopfield_async` descends. Sweep `frac` from 0 to 0.6 and record whether the final overlap with pattern 0 exceeds 0.8. Where does retrieval break, and why is that not a Necker cube?

<!-- solution-box: 05_1_corrupt_until_it_fails.py -->
<details>
<summary><strong>Show solution</strong> — Corrupt until it fails</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05_1_corrupt_until_it_fails.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.dynamics import corrupt, hebb_weights, hopfield_async, make_patterns, overlaps
patterns = make_patterns(n=32, seed=SEED)
W = hebb_weights(patterns)
fracs = np.linspace(0.0, 0.6, 13)
ok = []
for frac in fracs:
    cue = corrupt(patterns[0], frac=float(frac), seed=SEED + 3)
    recalled, _, _ = hopfield_async(cue, W, n_sweeps=8, seed=SEED)
    ov = float(overlaps(recalled, patterns)[0])
    ok.append(ov)
    print(f"frac {frac:.2f}  overlap with ξ0 {ov:+.2f}")

fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.plot(fracs, ok, "o-")
ax.axhline(0.8, color="0.6", ls="--", lw=1)
ax.set_xlabel("flip fraction")
ax.set_ylabel("final overlap with ξ0")
ax.set_title("A Hopfield basin has a radius; past it, recall fails")
plt.tight_layout()
plt.show()
print(
    "Inside the basin the corrupted cue is still closer to ξ0 than to ξ1, "
    "and asynchronous sign updates finish the job. Past a radius the cue "
    "is a different start and may fall into the other well or a mixture. "
    "That is associative memory, not bistable perception: there is no "
    "cube, and no viewer."
)
```

</details>
<!-- /solution-box -->


2. **β is the temperature.** `attention_retrieve(query, keys, beta)` is one modern-Hopfield step. Use the notebook’s mixed query ($0.55\,\xi_1+0.45\,\xi_2$) and print the softmax weights at $\beta \in \{0.5, 1, 4, 16\}$. When does the mass collapse onto one key, and why is that still not a transformer layer?

<!-- solution-box: 05_2_beta_is_the_temperature.py -->
<details>
<summary><strong>Show solution</strong> — β is the temperature</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05_2_beta_is_the_temperature.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.dynamics import attention_retrieve, make_patterns
keys = make_patterns(n=32, seed=SEED)
query = 0.55 * keys[0] + 0.45 * keys[1]
query = query / float(np.linalg.norm(query))
print(f"{'beta':>6}  w(ξ0)   w(ξ1)")
ws = []
for beta in (0.5, 1.0, 4.0, 16.0):
    _, w = attention_retrieve(query, keys, beta)
    ws.append(w)
    print(f"{beta:6.1f}  {w[0]:.3f}   {w[1]:.3f}")

fig, ax = plt.subplots(figsize=(4.6, 3.0))
ax.plot((0.5, 1, 4, 16), [w[0] for w in ws], "o-", label="ξ0")
ax.plot((0.5, 1, 4, 16), [w[1] for w in ws], "s-", label="ξ1")
ax.set_xlabel("β")
ax.set_ylabel("softmax weight")
ax.legend(frameon=False)
ax.set_title("Soft β shares; sharp β retrieves")
plt.tight_layout()
plt.show()
print(
    "Ramsauer et al.: one attention step with keys = values = stored "
    "patterns. Large β is a hard lookup, not a deeper network. Notebook "
    "01's maps are many such steps, trained, with a causal mask. This "
    "exercise is the *update*, not a language model and not a percept."
)
```

</details>
<!-- /solution-box -->


3. **Noise switches the well.** `necker_walk` is Langevin dynamics on $E(x)=\frac{{1}}{{4}}(x^2-1)^2$. Run two walks from $x_0=0$ with `sigma=0.05` and `sigma=0.55` (same seed). Count sign changes of a 40-step moving average. When is the state trapped, and when does it switch? What is still not a cube?

<!-- solution-box: 05_3_noise_switches_the_well.py -->
<details>
<summary><strong>Show solution</strong> — Noise switches the well</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05_3_noise_switches_the_well.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.dynamics import necker_walk
def n_switches(xs, win=40):
    ma = np.convolve(xs, np.ones(win) / win, mode="valid")
    return int(np.sum(np.diff(np.sign(ma)) != 0))

fig, axes = plt.subplots(2, 1, figsize=(6.0, 4.0), sharex=True)
for ax, sig in zip(axes, (0.05, 0.55)):
    xs = necker_walk(sigma=sig, x0=0.0, seed=SEED)
    ax.plot(xs, lw=0.8)
    ax.axhline(1, color="0.7", ls="--", lw=1)
    ax.axhline(-1, color="0.7", ls="--", lw=1)
    ax.set_ylabel("x")
    ax.set_title(f"σ = {sig}   switches ≈ {n_switches(xs)}", fontsize=9)
axes[-1].set_xlabel("step")
fig.suptitle("The same double well; only the noise scale changed")
plt.tight_layout()
plt.show()
print(
    "Small noise: the walk falls into one well and stays. Larger noise: "
    "it crosses the saddle. That is the teaching picture of bistable "
    "percepts as two attractors plus a perturbation. There is no cube, "
    "no rivalry experiment, and no claim that next-token sampling is "
    "seeing."
)
```

</details>
<!-- /solution-box -->
